# Project Cycle 3: Data Check & Cleaning (Extension Analysis)

**Dataset:** `YRBS_2007.csv` | **Target Audience:** US High School Students  
**Responsible Person:** 李宥宣
---

### Team Members

| Name | Student ID |
| :--- | :--- |
| **李宥宣** | `113370237` |
| **許皓崴** | `112370236` |

### Objective
The primary goal of this notebook is to perform **Data Cleaning** and **Variable Transformation** for our Extension Analysis (Mental Health vs. Sleep Hours). We focus on:
1. **Loading** the raw dataset without modification.
2. **Extracting** the approved variables for the extension (`SadOrHopeless` and `Sleep`).
3. **Handling** missing or invalid values.
4. **Transforming** the categorical sleep variable (codes 1-7) into a continuous numerical variable (4-10 hours) for the Two-Sample T-Test.
5. **Exporting** the processed data for reproducibility.

### Research Questions & Variable Definitions

**Independent Two-Sample T-Test Analysis**
* **Research Question:** Is there a significant difference in the average hours of sleep on school nights between students who felt sad or hopeless and those who did not?
* **Group Variable (Independent):** `SadOrHopeless` (Mental Health Status)
  * **Group 1:** code 1 (Felt sad or hopeless) $\rightarrow$ Recoded as label `'Sad / Hopeless'`
  * **Group 2:** code 2 (Did not feel sad or hopeless) $\rightarrow$ Recoded as label `'Not Sad'`
* **Response Variable (Dependent):** `Sleep` (Continuous variable)
  * **Transformation Rule:** Original code `1-7` is converted to actual hours by adding `3` (yielding `4-10` hours of sleep).

### Data Processing Pipeline

To ensure **reproducibility**, this cleaning script follows these steps:
1. **Load Data:** Read `../data/raw/YRBS_2007.csv`.
2. **Subset:** Extract only `SadOrHopeless` and `Sleep`.
3. **Handle Missing Data:** Count and drop rows with `NaN`.
4. **Transformation:** Convert the sleep codes into continuous numeric hours (`Sleep_Hours`).
5. **Recoding:** Map `SadOrHopeless` to clear text labels (`Sad_Label`) for high-quality Violin Plot visualization.
6. **Data Validation:** Check the summary statistics and final sample sizes.
7. **Export:** Save the final clean dataset to `../data/processed/extension_cleaned_data.csv`.

In [10]:
import pandas as pd
import numpy as np
import os

# ==========================================================
# 1. Environment Setup and Loading
# ==========================================================
raw_data_path = '../data/raw/YRBS_2007.csv'
df_raw = pd.read_csv(raw_data_path)
initial_count = len(df_raw)
print(f"✅ Raw data loaded successfully. Initial sample size: {initial_count}")

# ==========================================================
# 2. Variable Selection (Extension Analysis)
# ==========================================================
group_var = 'SadOrHopeless'
response_var = 'Sleep'

df_selected = df_raw[[group_var, response_var]].copy()

# ==========================================================
# 3. Handling Missing Values (NaN)
# ==========================================================
print("\n[Step 1] Missing Value Statistics:")
print(df_selected.isnull().sum().to_string())

df_cleaned = df_selected.dropna().copy()
after_dropna_count = len(df_cleaned)
print(f"\n[Step 2] Sample size after dropping missing values: {after_dropna_count}")

# ==========================================================
# 4. Variable Transformation & Recoding
# ==========================================================
# Sleep hours: Convert codes 1-7 to actual 4-10 hours
df_cleaned['Sleep_Hours'] = df_cleaned[response_var] + 3

# Mental Health: Map numeric codes to text labels for beautiful plotting
def label_sadness(code):
    if code == 1:
        return 'Sad / Hopeless'
    elif code == 2:
        return 'Not Sad'
    else:
        return np.nan

df_cleaned['Sad_Label'] = df_cleaned[group_var].apply(label_sadness)

# Drop any unexpected invalid rows generated during recoding
df_final = df_cleaned.dropna().copy()
final_count = len(df_final)

# ==========================================================
# 5. Data Validation & Export
# ==========================================================
os.makedirs('../data/processed', exist_ok=True)
processed_data_path = '../data/processed/extension_cleaned_data.csv'
df_final.to_csv(processed_data_path, index=False)

print("\n" + "="*50)
print(f"🏆 Final Data Cleaning Summary (Extension):")
print(f"   - Initial raw samples: {initial_count}")
print(f"   - Final usable samples: {final_count}")
print(f"   - Total attrition rate: {((initial_count - final_count) / initial_count * 100):.2f}%")
print("-" * 50)
print(f"   📊 Group Breakdown (Ready for T-test):")
print(f"      * Felt Sad / Hopeless: {df_final['Sad_Label'].value_counts().get('Sad / Hopeless', 0)}")
print(f"      * Not Sad:             {df_final['Sad_Label'].value_counts().get('Not Sad', 0)}")
print("="*50)
print(f"\n💾 Processed dataset saved to: {processed_data_path}")

✅ Raw data loaded successfully. Initial sample size: 14041

[Step 1] Missing Value Statistics:
SadOrHopeless     196
Sleep            1887

[Step 2] Sample size after dropping missing values: 12106

🏆 Final Data Cleaning Summary (Extension):
   - Initial raw samples: 14041
   - Final usable samples: 12106
   - Total attrition rate: 13.78%
--------------------------------------------------
   📊 Group Breakdown (Ready for T-test):
      * Felt Sad / Hopeless: 3581
      * Not Sad:             8525

💾 Processed dataset saved to: ../data/processed/extension_cleaned_data.csv


In [12]:
import os

# Define the comprehensive content for the reference document
reference_content = """# Project Cycle 3: Variable Notes (Extension Analysis)

**Dataset:** `YRBS_2007.csv`
**Research Question:** Extension (Mental Health and Sleep Hours)

## 1. Group Variable (Independent)
* **Original Variable:** `SadOrHopeless`
* **New Variable:** `Sad_Label`
* **Definition:** Used to divide the sample into two groups based on mental health status.
* **Recoding Rules:** * Code `1` (Yes) -> **`'Sad / Hopeless'`**
  * Code `2` (No) -> **`'Not Sad'`**

## 2. Response Variable (Dependent)
* **Original Variable:** `Sleep`
* **New Variable:** `Sleep_Hours`
* **Definition:** Continuous numeric variable representing the average hours of sleep on a school night.
* **Transformation Rules:**
  * Original codes ranged from `1` (4 or less hours) to `7` (10 or more hours).
  * Rule: **`Original Code + 3`** -> Yields a continuous range from **`4 to 10`** representing actual hours.
"""

reference_dir = '../references'
reference_file_path = os.path.join(reference_dir, 'extension_variable_notes.md')

os.makedirs(reference_dir, exist_ok=True)

with open(reference_file_path, 'w', encoding='utf-8') as f:
    f.write(reference_content)

print(f"✅ Reference documentation successfully created at: {reference_file_path}")

✅ Reference documentation successfully created at: ../references\extension_variable_notes.md
